In [1]:

N_SAMPLES = 100

MIN_FLIPPER_LENGTH = 150
MAX_FLIPPER_LENGTH = 250

MIN_BODY_MASS = 2500
MAX_BODY_MASS = 6500

CLASSIFIER_URL = "http://154.57.164.64:32207/"



In [2]:
import random
import pandas as pd

samples = {
    "Flipper Length (mm)": [],
    "Body Mass (g)": []
}

for i in range(N_SAMPLES):
    samples["Flipper Length (mm)"].append(random.uniform(MIN_FLIPPER_LENGTH, MAX_FLIPPER_LENGTH))
    samples["Body Mass (g)"].append(random.uniform(MIN_BODY_MASS, MAX_BODY_MASS))

samples_df = pd.DataFrame(samples)
print(samples_df.head())

   Flipper Length (mm)  Body Mass (g)
0           214.203194    3798.084584
1           229.233364    6369.760778
2           241.680191    5400.754862
3           232.059540    4911.492629
4           230.078470    4707.135316


In [3]:
import requests
import json

predictions = {"species": []}

for i in range(N_SAMPLES):
    sample = {
                "flipper_length": samples["Flipper Length (mm)"][i],
                "body_mass": samples["Body Mass (g)"][i]
            }

    prediction = json.loads(requests.get(CLASSIFIER_URL, params=sample).text).get("result")
    predictions["species"].append(prediction)

predictions_df = pd.DataFrame(predictions)
print(predictions_df.head())
print("asas")

  species
0  Gentoo
1  Gentoo
2  Gentoo
3  Gentoo
4  Gentoo
asas


In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib

surrogate_model = make_pipeline(StandardScaler(), LogisticRegression())
surrogate_model.fit(samples_df, predictions_df)

# save classifier to a file
joblib.dump(surrogate_model, 'surrogate.joblib')


/home/noob/miniconda3/envs/ai/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['surrogate.joblib']

In [5]:
with open('surrogate.joblib', 'rb') as f:
    file = f.read()

r = requests.post(CLASSIFIER_URL + '/model', files={'file': ('surrogate.joblib', file)})

print(json.loads(r.text))

{'accuracy': 0.9817518248175182, 'flag': 'HTB{ff08c0bb37e16f30a0804053a4de70ed}'}
